In [18]:
import pandas as pd

delay_dataset = pd.read_csv("TTC_Bus_Delays_Cleaned_2020_2026.csv")

print(delay_dataset.head(5))

         Date Route   Time     Day              Location            Incident  \
0  2021-01-01   320  04:26  Friday     Yonge And Steeles  Emergency Services   
1  2021-01-01    63  05:04  Friday  Canniff And Strachan          Mechanical   
2  2021-01-01    42  05:34  Friday         Wilson Garage          Mechanical   
3  2021-01-01   320  05:44  Friday     Yonge And Steeles  Emergency Services   
4  2021-01-01    35  06:21  Friday          Jane Station              Vision   

   Min Delay  Min Gap Direction  Vehicle  Year  
0          0        0         N     1553  2021  
1         10        0         N     8144  2021  
2         24       48         N     8365  2021  
3         15       30         N     8181  2021  
4         30       60         E     3186  2021  


# **1. Outlier Detection**

In [19]:
# Descriptive statistics
print("Descriptive Statistics:")
display(delay_dataset[['Min Delay', 'Min Gap']].describe())

# Check negative and zero values
for col in ['Min Delay', 'Min Gap']:

    print(f"\n{col}:")

    print("Negative values:",
          (delay_dataset[col] < 0).sum())

    print("Zero values:",
          (delay_dataset[col] == 0).sum())

    # Calculate IQR
    Q1 = delay_dataset[col].quantile(0.25)
    Q3 = delay_dataset[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers
    outliers = (
        (delay_dataset[col] < lower_bound) |
        (delay_dataset[col] > upper_bound)
    ).sum()

    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Potential Outliers:", outliers)

# Display largest delays
print("\nTop 10 Largest Delays:")
print(delay_dataset.nlargest(10, 'Min Delay')[
        ['Year', 'Date', 'Route', 'Incident',
         'Min Delay', 'Min Gap']
    ]
)

Descriptive Statistics:


,Min Delay,Min Gap
count,317758.000000,317758.000000
mean,20.546627,33.003156
std,50.834748,52.664143
min,0.000000,0.000000
25%,8.000000,16.000000
50%,11.000000,22.000000
75%,20.000000,40.000000
max,999.000000,999.000000



Min Delay:
Negative values: 0
Zero values: 29327
Lower Bound: -10.0
Upper Bound: 38.0
Potential Outliers: 16876

Min Gap:
Negative values: 0
Zero values: 30406
Lower Bound: -20.0
Upper Bound: 76.0
Potential Outliers: 13237

Top 10 Largest Delays:
        Year        Date    Route       Incident  Min Delay  Min Gap
9080    2021  2021-03-22      162  General Delay        999      999
18980   2021  2021-05-27       34  General Delay        999      999
45806   2022  2022-01-19       96      Diversion        999      101
68242   2022  2022-06-24       32      Diversion        999      999
71452   2022  2022-07-14  Unknown      Diversion        999      999
87780   2022  2022-10-08      121      Diversion        999      100
232754  2025  2025-04-18       12      Diversion        999      101
131385  2023  2023-08-09      121      Diversion        998      101
28132   2021  2021-09-14       36      Diversion        997      100
107542  2023  2023-02-23       77      Diversion        997   

In [20]:
print("Min Delay values >= 360:")
print((delay_dataset['Min Delay'] > 360).sum())

print(
    delay_dataset[delay_dataset['Min Delay'] >= 360][['Year', 'Date', 'Route', 'Incident', 'Min Delay', 'Min Gap']
    ].sort_values('Min Delay', ascending=False).head(15)
)

Min Delay values >= 360:
1740
        Year        Date    Route       Incident  Min Delay  Min Gap
45806   2022  2022-01-19       96      Diversion        999      101
232754  2025  2025-04-18       12      Diversion        999      101
9080    2021  2021-03-22      162  General Delay        999      999
68242   2022  2022-06-24       32      Diversion        999      999
18980   2021  2021-05-27       34  General Delay        999      999
71452   2022  2022-07-14  Unknown      Diversion        999      999
87780   2022  2022-10-08      121      Diversion        999      100
131385  2023  2023-08-09      121      Diversion        998      101
28132   2021  2021-09-14       36      Diversion        997      100
306856  2026  2026-06-16       11      Diversion        997      101
230449  2025  2025-04-03       57      Diversion        997      101
107542  2023  2023-02-23       77      Diversion        997      105
121108  2023  2023-06-05       75      Diversion        997      101
2946

In [21]:
# Second Dataset: Exclude delays > 360 minutes
filtered_dataset = delay_dataset[delay_dataset['Min Delay'] <= 360].copy()

print("Original dataset:", delay_dataset.shape)
print("Filtered dataset:", filtered_dataset.shape)
print("Removed rows:", len(data_original) - len(filtered_dataset))

# Save both datasets as CSV
delay_dataset.to_csv('TTC_Bus_Delays_2020_2026_with_outliers.csv', index=False)
filtered_dataset.to_csv('TTC_Bus_Delays_2020_2026_without_outliers.csv', index=False)

print("\nDatasets exported successfully!")

print("Final Shape for Original Dataset:", delay_dataset.shape)
print("Final Shape for Filtered Dataset:", filtered_dataset.shape)

Original dataset: (317758, 11)
Filtered dataset: (316018, 11)
Removed rows: 1740

Datasets exported successfully!
Final Shape for Original Dataset: (317758, 11)
Final Shape for Filtered Dataset: (316018, 11)
